# Enumerate-then-sample vs Direct Random Sampling

Compares two strategies for selecting random guide sets:
- **enumerate**: enumerate all frontier terms, then sample random subsets from those
- **random**: directly sample frontier terms

Only looks at pure random guide selection (no ZS/structural ranking, no helpers).

In [ ]:
import json

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import find_latest_run, parse_random_trials

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
enum_dir = find_latest_run("enumerate")
rand_dir = find_latest_run("random")

print(f"Enumerate: {enum_dir}")
print(f"Random:    {rand_dir}")

with open(enum_dir / "top_k.json") as f:
    raw_enum = json.load(f)
with open(rand_dir / "top_k.json") as f:
    raw_rand = json.load(f)

print(f"Enumerate: {len(raw_enum)} goal(s)")
print(f"Random:    {len(raw_rand)} goal(s)")

In [ ]:
# ── Parse into flat DataFrames ─────────────────────────────────────────
rows_enum = parse_random_trials(raw_enum, "enumerate", "random")
rows_rand = parse_random_trials(raw_rand, "random", "entries")

df = pl.DataFrame(rows_enum + rows_rand)
k_values = sorted(df["k"].unique().to_list())

print(f"Total trial rows: {len(df)}")
print(f"  enumerate: {len(rows_enum)}")
print(f"  random:    {len(rows_rand)}")
print(f"k values: {k_values}")
df.head(5)

## Combined iterations vs k

In [ ]:
COLORS = {"enumerate": "steelblue", "random": "coral"}
MARKERS = {"enumerate": "o", "random": "s"}
STRATEGIES = ["enumerate", "random"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, ax in zip(["combined_iters", "combined_nodes"], axes):
    for strat in STRATEGIES:
        agg = (
            df.filter(pl.col("strategy") == strat)
            .group_by("k")
            .agg(
                pl.col(metric).mean().alias("mean"),
                pl.col(metric).std().alias("std"),
            )
            .sort("k")
        )
        ks = agg["k"].to_numpy()
        means = agg["mean"].to_numpy()
        stds = agg["std"].fill_null(0).to_numpy()
        ax.plot(ks, means, marker=MARKERS[strat], color=COLORS[strat], label=strat)
        ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=COLORS[strat])

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs k (mean \u00b1 std over goals)")
    ax.legend()
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

fig.tight_layout()
plt.show()

## Reachability rate vs k

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for strat in STRATEGIES:
    reach = (
        df.filter(pl.col("strategy") == strat)
        .group_by("k")
        .agg(pl.col("combined_iters").is_not_null().mean().alias("reach_rate"))
        .sort("k")
    )
    ax.plot(
        reach["k"].to_numpy(),
        reach["reach_rate"].to_numpy() * 100,
        marker=MARKERS[strat],
        color=COLORS[strat],
        label=strat,
    )

ax.set_xlabel("k (number of guides)")
ax.set_ylabel("reachability (%)")
ax.set_title("Fraction of trials reaching the goal vs k")
ax.legend()
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
ax.set_ylim(-5, 105)
fig.tight_layout()
plt.show()

## Box plots: combined iterations by strategy at each k

In [ ]:
n_k = len(k_values)
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for strat in STRATEGIES:
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))[
                "combined_iters"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(COLORS[strat])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("combined_iters" if ki == 0 else "")

fig.suptitle("Combined iterations by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Box plots: combined nodes by strategy at each k

In [ ]:
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for strat in STRATEGIES:
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))[
                "combined_nodes"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(COLORS[strat])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("combined_nodes" if ki == 0 else "")

fig.suptitle("Combined nodes by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Best single guide iters vs combined (per strategy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, strat in zip(axes, STRATEGIES):
    sub = df.filter(pl.col("strategy") == strat).drop_nulls(
        subset=["best_single_iters", "combined_iters"]
    )
    if len(sub) == 0:
        ax.set_visible(False)
        continue
    x = sub["best_single_iters"].to_numpy()
    y = sub["combined_iters"].to_numpy()
    ks = sub["k"].to_numpy()
    scatter = ax.scatter(
        x,
        y,
        c=np.log2(ks),
        cmap="viridis",
        s=20,
        alpha=0.4,
        edgecolors="k",
        linewidths=0.3,
    )
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("log2(k)")
    lims = [min(x.min(), y.min()) - 0.5, max(x.max(), y.max()) + 0.5]
    ax.plot(lims, lims, "k--", alpha=0.3, lw=1)
    ax.set_xlabel("best_single_iters")
    ax.set_ylabel("combined_iters")
    ax.set_title(f"{strat}: single vs combined")
    ax.set_aspect("equal")

fig.suptitle("Best single guide iters vs combined top-k iters", fontsize=13)
fig.tight_layout()
plt.show()

## Improvement from combining: (best_single - combined) vs k

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for strat in STRATEGIES:
    sub = (
        df.filter(pl.col("strategy") == strat)
        .drop_nulls(subset=["best_single_iters", "combined_iters"])
        .with_columns(
            (pl.col("best_single_iters") - pl.col("combined_iters")).alias(
                "improvement"
            )
        )
    )
    agg = (
        sub.group_by("k")
        .agg(
            pl.col("improvement").mean().alias("mean"),
            pl.col("improvement").std().alias("std"),
        )
        .sort("k")
    )
    ks = agg["k"].to_numpy()
    means = agg["mean"].to_numpy()
    stds = agg["std"].fill_null(0).to_numpy()
    ax.plot(ks, means, marker=MARKERS[strat], color=COLORS[strat], label=strat)
    ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=COLORS[strat])

ax.axhline(0, color="black", ls=":", alpha=0.3)
ax.set_xlabel("k")
ax.set_ylabel("improvement (best_single - combined iters)")
ax.set_title("Iteration improvement from combining guides")
ax.legend()
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
fig.tight_layout()
plt.show()

## Summary table

In [ ]:
summary = (
    df.group_by("strategy", "k")
    .agg(
        pl.col("combined_iters").mean().round(2).alias("avg_combined_iters"),
        pl.col("combined_nodes").mean().round(0).alias("avg_combined_nodes"),
        pl.col("best_single_iters").mean().round(2).alias("avg_best_single_iters"),
        pl.col("combined_iters").is_not_null().mean().round(3).alias("reachability"),
        pl.col("combined_iters").count().alias("n_trials"),
    )
    .sort("k", "strategy")
)
summary